In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import re
import time

In [33]:
category=input('Category=')
category

Category= rabbit


'rabbit'

In [37]:
# Initialize WebDriver
driver = webdriver.Chrome()
driver.maximize_window()

pet_data = []  # Store scraped data


file_name = category
file_name = file_name.strip() + ".xlsx" 
directory = "D:/scrap/"
full_path = directory + file_name 

base_url = "https://katabononline.com/product-category/"
url = base_url + category

# Get total pages dynamically
driver.get(url)
time.sleep(3)

pagination_text = driver.find_element(By.XPATH, "//p[@class='woocommerce-result-count hide-for-medium']").text

# Check which format is present
if "Showing 1–" in pagination_text:  # ✅ Format: "Showing 1–100 of 236 results"
    total_products = int(re.search(r'of (\d+)', pagination_text).group(1))

elif "Showing all" in pagination_text:  # ✅ Format: "Showing all 64 results"
    total_products = int(re.search(r'all (\d+)', pagination_text).group(1))

elif "Showing the single result" in pagination_text:  # ✅ Format: "Showing the single result"
    total_products = 1  # ✅ Only one product available

else:
    total_products = 0  # ✅ Default case (if format is unknown)



total_pages = -(-total_products // 100)


print(f"Total Pages Detected: {total_pages}")
print(f"Total Products Found: {total_products}")




page = 1
while page <= total_pages:
    driver.get(f"{url}/page/{page}")
    time.sleep(2)  # Allow page to load properly

    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//div[@class='shop-container']")))

    # Determine iteration count based on page number
    if page < total_pages:
        product_count = 100  # First few pages always have 100 products
    else:
        product_count = total_products % 100  # Last page has remaining products

    print(f"Scraping {product_count} products from page {page}...")

    # for i in range(1, product_count + 1):  # Iterate from 1 to product_count
    #     product_xpath = f"//div[@class='mb-7']//div[@class='col'][{i}]"  # Select product dynamically

    #     try:
    #         product = WebDriverWait(driver, 5).until(
    #             EC.presence_of_element_located((By.XPATH, product_xpath))
    #         )
    #         # product.click()    #it will click products
    #     except Exception as e:
    #         print(f"Skipping product {i} due to error: {e}")
    #         continue

        # time.sleep(2)

    

    product_elements = driver.find_elements(By.XPATH, "//div[@class='product-small box ']")

    for product in product_elements[:product_count]:  # ✅ Loop through actual products
        try:
            try:
                product_name = product.find_element(By.XPATH,".//a[@class='woocommerce-LoopProduct-link woocommerce-loop-product__link']").text   #adding . before //h5 will select only that product
            except:
                product_name = ""
            
            try:
                mrp = product.find_element(By.XPATH, ".//del//bdi").text
            except:
                mrp = product.find_element(By.XPATH, ".//span[@class='price']/span/bdi").text
                
            try:
                prc = product.find_element(By.XPATH, ".//ins//bdi").text
            except:
                prc = product.find_element(By.XPATH, ".//span[@class='price']/span/bdi").text
            
            # try:
            #     stock =  product.find_element(By.XPATH,".//span[@class='fw-700 fs-13 opacity-60']").text
            # except:
            #     stock = "In Stock"
    
            
            mrp1 = mrp.replace("৳", "").replace(",", "").split(".")[0].strip()
            prc1 = prc.replace("৳", "").replace(",", "").split(".")[0].strip()
            
    
            pet_data.append({
                "Name": product_name,
                "MRP": mrp1,
                "Discounted prc": prc1
                # "Stock": stock
            })

        except Exception as e:
            print(f"Error while scraping product: {e}")

        # driver.back()  
        # time.sleep(2)

    if page % 1 == 0:
        df = pd.DataFrame(pet_data)
        backup_file = f"{directory}{category.strip()}_backup.xlsx"
        df.to_excel(backup_file, index=False, engine="openpyxl")
        print(f"✅ Backup saved at page {page}: {backup_file}")
    
    page += 1  



df = pd.DataFrame(pet_data)
print(df)


df.to_excel(full_path, index=False, engine='openpyxl')
print(f"File saved successfully at {full_path}")

driver.quit()

Total Pages Detected: 1
Total Products Found: 19
Scraping 19 products from page 1...
✅ Backup saved at page 1: D:/scrap/rabbit_backup.xlsx
                                                 Name   MRP Discounted prc
0   Round Plush Pet Bed Super Soft and Super Warm ...  3500           2999
1   Round Plush Pet Bed Super Soft and Super Warm ...  3500           2999
2   Round Plush Pet Bed Super Soft and Super Warm ...  3500           2999
3       Portable Folding Pet Tent House For Dog & Cat  3650           2990
4   Round Plush Pet Bed Super Soft and Super Warm ...  1800           1450
5   Round Plush Pet Bed Super Soft and Super Warm ...  1800           1450
6   Cat Bag Breathable Portable Pet Carrier Bag Ou...  2100           1400
7                            Square Pet House For Pet  1250           1050
8                       RFL Pet Carrier Family Basket   800            550
9                         Coat care glove duo 24x19cm   750            550
10  Nutrich Tabs Complete multivitam